# Manual Trading Round 2

This notebook explores the Round 2 allocation problem across **Research**, **Scale**, and **Speed**.

Assumption used here:
- The investment input `x` in the prompt is treated as a **percentage of the total budget**.
- A value of `100` means allocating the full `50_000` budget to that pillar.
- Net PnL is calculated as `research(x_r) * scale(x_s) * speed_multiplier - used_budget`.
- The heatmap only shows **feasible** allocations where `research + scale + speed <= 100`.

The speed outcome is rank-based in the real game, so this notebook exposes the **speed multiplier** as a scenario input. It now includes a second heatmap for **speed investment vs speed multiplier**, where each cell shows the best net PnL after optimizing Research and Scale for that speed scenario.

In [ ]:
import uuid

import plotly.offline as pyo
from IPython.display import HTML, display

TOTAL_BUDGET = 50_000
DEFAULT_SPEED_INVESTMENT = 20
DEFAULT_SPEED_MULTIPLIER = 0.50
GRID_VALUES = list(range(101))
MULTIPLIER_VALUES = [round(value / 100, 2) for value in range(10, 91)]

container_id = f"round2-allocation-{uuid.uuid4().hex}"
plotly_js = pyo.get_plotlyjs()
grid_js = str(GRID_VALUES)
multiplier_grid_js = str(MULTIPLIER_VALUES)

html = f"""
<div id=\"{container_id}\" style=\"font-family: Arial, sans-serif; max-width: 1200px;\">
  <div style=\"margin-bottom: 16px;\">
    <label for=\"{container_id}-speed\" style=\"display: block; font-weight: 600; margin-bottom: 6px;\">Speed investment (% of total budget)</label>
    <input id=\"{container_id}-speed\" type=\"range\" min=\"0\" max=\"100\" step=\"1\" value=\"{DEFAULT_SPEED_INVESTMENT}\" style=\"width: 100%;\" />
    <div id=\"{container_id}-speed-value\" style=\"margin-top: 4px; color: #444;\"></div>
  </div>
  <div style=\"margin-bottom: 16px;\">
    <label for=\"{container_id}-multiplier\" style=\"display: block; font-weight: 600; margin-bottom: 6px;\">Speed multiplier</label>
    <input id=\"{container_id}-multiplier\" type=\"range\" min=\"0.10\" max=\"0.90\" step=\"0.01\" value=\"{DEFAULT_SPEED_MULTIPLIER:.2f}\" style=\"width: 100%;\" />
    <div id=\"{container_id}-multiplier-value\" style=\"margin-top: 4px; color: #444;\"></div>
  </div>
  <div style=\"font-size: 1.05rem; font-weight: 700; margin: 20px 0 8px;\">Research vs Scale for the current speed settings</div>
  <div id=\"{container_id}-research-scale-plot\" style=\"min-height: 720px;\"></div>
  <div style=\"font-size: 1.05rem; font-weight: 700; margin: 28px 0 8px;\">Best net PnL across speed scenarios</div>
  <div id=\"{container_id}-speed-scenario-plot\" style=\"min-height: 650px;\"></div>
  <div id=\"{container_id}-summary\" style=\"margin-top: 16px; padding: 14px 16px; background: #f5f7fb; border: 1px solid #d8e0ef; border-radius: 10px;\"></div>
</div>
<script type=\"text/javascript\">{plotly_js}</script>
<script type=\"text/javascript\">
(function() {{
  const budget = {TOTAL_BUDGET};
  const grid = {grid_js};
  const multiplierGrid = {multiplier_grid_js};
  const speedSlider = document.getElementById('{container_id}-speed');
  const multiplierSlider = document.getElementById('{container_id}-multiplier');
  const speedValue = document.getElementById('{container_id}-speed-value');
  const multiplierValue = document.getElementById('{container_id}-multiplier-value');
  const researchScalePlotDiv = document.getElementById('{container_id}-research-scale-plot');
  const speedScenarioPlotDiv = document.getElementById('{container_id}-speed-scenario-plot');
  const summaryDiv = document.getElementById('{container_id}-summary');

  const researchValues = grid.map((x) => (x <= 0 ? 0 : 200000 * Math.log(1 + x) / Math.log(101)));
  const scaleValues = grid.map((x) => 7 * x / 100);

  function usedBudget(researchPct, scalePct, speedPct) {{
    return budget * (researchPct + scalePct + speedPct) / 100;
  }}

  function formatInt(value) {{
    return value.toLocaleString(undefined, {{ maximumFractionDigits: 0 }});
  }}

  function formatPct(value) {{
    return `${{value.toFixed(0)}}%`;
  }}

  function computeScenario(speedPct, speedMultiplier, includeSurface) {{
    const cap = 100 - speedPct;
    const z = includeSurface ? [] : null;
    let best = {{
      pnl: Number.NEGATIVE_INFINITY,
      gross: 0,
      researchPct: 0,
      scalePct: 0,
      used: budget * speedPct / 100,
    }};

    for (let researchPct = 0; researchPct <= 100; researchPct += 1) {{
      const maxScale = cap - researchPct;

      if (includeSurface) {{
        const row = new Array(101).fill(null);

        if (maxScale >= 0) {{
          for (let scalePct = 0; scalePct <= maxScale; scalePct += 1) {{
            const gross = researchValues[researchPct] * scaleValues[scalePct] * speedMultiplier;
            const used = usedBudget(researchPct, scalePct, speedPct);
            const pnl = gross - used;
            row[scalePct] = pnl >= 0 ? pnl : null;

            if (pnl > best.pnl) {{
              best = {{
                pnl,
                gross,
                researchPct,
                scalePct,
                used,
              }};
            }}
          }}
        }}

        z.push(row);
      }} else if (maxScale >= 0) {{
        for (let scalePct = 0; scalePct <= maxScale; scalePct += 1) {{
          const gross = researchValues[researchPct] * scaleValues[scalePct] * speedMultiplier;
          const used = usedBudget(researchPct, scalePct, speedPct);
          const pnl = gross - used;

          if (pnl > best.pnl) {{
            best = {{
              pnl,
              gross,
              researchPct,
              scalePct,
              used,
            }};
          }}
        }}
      }}
    }}

    return {{ z, best }};
  }}

  function isMaskedSpeedScenario(speedPct, speedMultiplier) {{
    if (speedPct < 0 || speedPct > 40 || speedMultiplier < 0.20 || speedMultiplier > 0.90) {{
      return false;
    }}

    const boundaryMultiplier = 0.20 + (0.70 * speedPct / 40);
    return speedMultiplier >= boundaryMultiplier;
  }}

  function buildSpeedScenarioSurface() {{
    const z = [];

    for (let multiplierIndex = 0; multiplierIndex < multiplierGrid.length; multiplierIndex += 1) {{
      const speedMultiplier = multiplierGrid[multiplierIndex];
      const row = [];

      for (let speedPct = 0; speedPct <= 100; speedPct += 1) {{
        if (isMaskedSpeedScenario(speedPct, speedMultiplier)) {{
          row.push(null);
          continue;
        }}

        const bestPnl = computeScenario(speedPct, speedMultiplier, false).best.pnl;
        row.push(bestPnl >= 0 ? bestPnl : null);
      }}

      z.push(row);
    }}

    return z;
  }}

  const speedScenarioZ = buildSpeedScenarioSurface();

  function render() {{
    const speedPct = Number(speedSlider.value);
    const speedMultiplier = Number(multiplierSlider.value);
    const speedBudget = budget * speedPct / 100;
    const feasibleCap = Math.max(0, 100 - speedPct);
    const {{ z, best }} = computeScenario(speedPct, speedMultiplier, true);

    speedValue.textContent = `${{formatPct(speedPct)}} of budget (${{formatInt(speedBudget)}} units)`;
    multiplierValue.textContent = speedMultiplier.toFixed(2);

    const researchScaleHeatmap = {{
      type: 'heatmap',
      x: grid,
      y: grid,
      z,
      colorscale: 'Turbo',
      colorbar: {{ title: 'Net PnL' }},
      hovertemplate: 'Scale %{{x}}%<br>Research %{{y}}%<br>Net PnL %{{z:,.0f}}<extra></extra>',
    }};

    const bestPoint = {{
      type: 'scatter',
      mode: 'markers+text',
      x: [best.scalePct],
      y: [best.researchPct],
      text: ['Best'],
      textposition: 'top center',
      marker: {{
        size: 14,
        color: '#ff6b6b',
        line: {{ color: '#ffffff', width: 1.5 }},
      }},
      hovertemplate: `Best point<br>Scale ${{best.scalePct}}%<br>Research ${{best.researchPct}}%<br>Net PnL ${{formatInt(best.pnl)}}<extra></extra>`,
      showlegend: false,
    }};

    const researchScaleLayout = {{
      template: 'plotly_white',
      title: `Round 2 net PnL | Speed = ${{speedPct}}% | Multiplier = ${{speedMultiplier.toFixed(2)}}`,
      xaxis: {{ title: 'Scale investment (% of total budget)', dtick: 10, range: [0, 100] }},
      yaxis: {{ title: 'Research investment (% of total budget)', dtick: 10, range: [0, 100] }},
      margin: {{ l: 75, r: 30, t: 70, b: 70 }},
      height: 720,
      shapes: [
        {{
          type: 'line',
          x0: 0,
          y0: feasibleCap,
          x1: feasibleCap,
          y1: 0,
          line: {{ color: '#ffffff', dash: 'dash', width: 2 }},
        }},
      ],
      annotations: [
        {{
          x: feasibleCap / 2,
          y: Math.min(98, feasibleCap / 2 + 8),
          text: 'Budget frontier',
          showarrow: false,
          font: {{ color: '#ffffff' }},
        }},
      ],
    }};

    Plotly.react(researchScalePlotDiv, [researchScaleHeatmap, bestPoint], researchScaleLayout, {{ responsive: true, displayModeBar: false }});

    const speedScenarioHeatmap = {{
      type: 'heatmap',
      x: grid,
      y: multiplierGrid,
      z: speedScenarioZ,
      colorscale: 'Turbo',
      colorbar: {{ title: 'Best net PnL' }},
      hovertemplate: 'Speed %{{x}}%<br>Multiplier %{{y:.2f}}<br>Best net PnL %{{z:,.0f}}<extra></extra>',
    }};

    const currentScenarioPoint = {{
      type: 'scatter',
      mode: 'markers+text',
      x: [speedPct],
      y: [speedMultiplier],
      text: ['Current'],
      textposition: 'top center',
      marker: {{
        size: 14,
        color: '#ef476f',
        line: {{ color: '#ffffff', width: 1.5 }},
      }},
      hovertemplate: `Current scenario<br>Speed ${{speedPct}}%<br>Multiplier ${{speedMultiplier.toFixed(2)}}<br>Best net PnL ${{formatInt(best.pnl)}}<extra></extra>`,
      showlegend: false,
    }};

    const speedScenarioLayout = {{
      template: 'plotly_white',
      title: 'Best net PnL after optimizing Research and Scale',
      xaxis: {{ title: 'Speed investment (% of total budget)', dtick: 10, range: [0, 100] }},
      yaxis: {{ title: 'Speed multiplier', tickformat: '.2f', range: [0.10, 0.90] }},
      margin: {{ l: 75, r: 30, t: 70, b: 70 }},
      height: 650,
    }};

    Plotly.react(speedScenarioPlotDiv, [speedScenarioHeatmap, currentScenarioPoint], speedScenarioLayout, {{ responsive: true, displayModeBar: false }});

    const remainingBudget = budget - best.used;
    summaryDiv.innerHTML = `
      <div style="font-size: 1.05rem; font-weight: 700; margin-bottom: 8px;">Best net PnL with current speed settings: ${{formatInt(best.pnl)}}</div>
      <div><strong>Research allocation:</strong> ${{formatPct(best.researchPct)}} (${{formatInt(budget * best.researchPct / 100)}} units)</div>
      <div><strong>Scale allocation:</strong> ${{formatPct(best.scalePct)}} (${{formatInt(budget * best.scalePct / 100)}} units)</div>
      <div><strong>Speed allocation:</strong> ${{formatPct(speedPct)}} (${{formatInt(speedBudget)}} units)</div>
      <div><strong>Speed multiplier:</strong> ${{speedMultiplier.toFixed(2)}}</div>
      <div><strong>Gross PnL before budget deduction:</strong> ${{formatInt(best.gross)}}</div>
      <div><strong>Total used budget:</strong> ${{formatInt(best.used)}}</div>
      <div><strong>Unused budget:</strong> ${{formatInt(remainingBudget)}}</div>
    `;
  }}

  speedSlider.addEventListener('input', render);
  multiplierSlider.addEventListener('input', render);
  render();
}})();
</script>
"""

display(HTML(html))


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "manual_trading").exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break

from manual_trading.round2_manual_trading_simulator import display_round2_manual_trading_simulator

display_round2_manual_trading_simulator()


## 20,000-player sampled speed field

This graph samples 5,000 players from each speed-investment cluster: `N(25%, variance 3)`, `N(37%, variance 3)`, `N(60%, variance 3)`, and `Uniform(23%, 63%)`. Samples are rounded to integer speed percentages, then the same Round 2 research/scale optimizer is used to compute PnL by speed.

In [ ]:
from manual_trading.round2_speed_investment_pnl import display_clustered_speed_investment_pnl_graph

players_20000, pnl_by_speed, speed_pnl_fig = display_clustered_speed_investment_pnl_graph(seed=42)

pnl_by_speed.sort_values("pnl", ascending=False).head(10)
